# Semantic Embedding-Based Search

Semantic search uses sentence embeddings to understand document meaning and retrieve semantically similar documents. This implementation encodes documents and queries into vector embeddings, then uses FAISS for efficient nearest-neighbor search. The approach captures semantic similarity beyond exact keyword matching. Output: `data/processed/semantic_index`

**Workflow:**

1. Install sentence-transformers and FAISS libraries
2. Implement SemanticRetriever class (`src/semantic.py`)
3. Build semantic index from corpus
4. Test search on sample queries

## Prerequisites Check

First run `01_data_preparation.ipynb` to generate `corpus.pkl`, `books_sample.parquet`, and `utils.py`.

In [1]:
import os
from pathlib import Path

notebook_dir = Path.cwd()
project_root = notebook_dir.parent if 'notebooks' in str(notebook_dir) else notebook_dir

prereqs = [
    project_root / 'data/processed/corpus.pkl',
    project_root / 'src/utils.py'
]

print("Checking prerequisites:")
for prereq in prereqs:
    exists = os.path.exists(prereq)
    status = "OK" if exists else "MISSING"
    print(f"  {status}: {prereq}")

if all(os.path.exists(p) for p in prereqs):
    print("\nAll prerequisites ready. Continue below.")
else:
    print("\nRun 01_eda.ipynb first!")

Checking prerequisites:
  OK: /Users/esteki/Desktop/MDS/575/DSCI_575_project_jchuang_esteki/data/processed/corpus.pkl
  OK: /Users/esteki/Desktop/MDS/575/DSCI_575_project_jchuang_esteki/src/utils.py

All prerequisites ready. Continue below.


## 1. Install Dependencies

Install sentence-transformers (neural text embeddings) and faiss-cpu (fast nearest-neighbour vector search).

In [2]:
import subprocess, sys

for package, import_name in [("sentence-transformers", "sentence_transformers"), ("faiss-cpu", "faiss")]:
    try:
        __import__(import_name)
        print(f"{package} already installed")
    except ImportError:
        print(f"Installing {package}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", package, "-q"])
        print("Done")

sentence-transformers already installed
faiss-cpu already installed


## 2. Setup Project Path

Add the project root to `sys.path` so we can import `src/semantic.py`.

In [3]:
from pathlib import Path
import sys

notebook_dir = Path.cwd()
if 'notebooks' in str(notebook_dir):
    project_root = notebook_dir.parent
else:
    project_root = notebook_dir

sys.path.insert(0, str(project_root))

print(f"Project root: {project_root}")
print(f"src/semantic.py exists: {(project_root / 'src' / 'semantic.py').exists()}")

Project root: /Users/esteki/Desktop/MDS/575/DSCI_575_project_jchuang_esteki
src/semantic.py exists: True


## 3. Build Semantic Index

Load the corpus, encode all documents into 384-dimensional vectors using `all-MiniLM-L6-v2`, and store them in a FAISS index. This is the slow step (~2–5 min for 20K documents). Output: `data/processed/faiss_index`

In [4]:
from src.semantic import SemanticRetriever
from src.utils import load_corpus

corpus_file = str(project_root / 'data' / 'processed' / 'corpus.pkl')
index_file = str(project_root / 'data' / 'processed' / 'semantic_index')

corpus = load_corpus(corpus_file)
print(f"Loaded corpus: {len(corpus)} documents\n")

semantic_retriever = SemanticRetriever()

if Path(index_file).exists():
    print(f"Index already exists at {index_file}, skipping build.")
    semantic_retriever.load(index_file)
else:
    print("Building semantic index (this may take a few minutes)...")
    semantic_retriever.build_index(corpus)
    print(f"\nSaving to: {index_file}")
    semantic_retriever.save(index_file)
    print("Semantic index complete")

Loaded corpus: 20000 documents



Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Building semantic index (this may take a few minutes)...
Encoding 20000 documents...


Batches:   0%|          | 0/625 [00:00<?, ?it/s]

FAISS index built with 20000 vectors (dim=384)

Saving to: /Users/esteki/Desktop/MDS/575/DSCI_575_project_jchuang_esteki/data/processed/semantic_index
FAISS index saved to /Users/esteki/Desktop/MDS/575/DSCI_575_project_jchuang_esteki/data/processed/semantic_index
Semantic index complete


## 4. Test Semantic Search

Load the persisted FAISS index and test on the same sample queries used in the BM25 notebook. Note: distances are L2 (lower = more similar), unlike BM25 scores (higher = more relevant).

In [5]:
import pandas as pd
from src.semantic import SemanticRetriever
from src.utils import load_corpus

index_file = str(project_root / 'data' / 'processed' / 'semantic_index')
corpus_file = str(project_root / 'data' / 'processed' / 'corpus.pkl')

semantic_retriever = SemanticRetriever()
semantic_retriever.load(index_file)
corpus = load_corpus(corpus_file)
semantic_retriever.corpus = corpus

# Load data for result enrichment
df = pd.read_parquet(project_root / 'data' / 'processed' / 'books_sample.parquet')
lookup = pd.read_parquet(project_root / 'data' / 'processed' / 'asin_title_lookup.parquet') \
           .set_index('parent_asin')['book_title'].to_dict()

test_queries = ["python book", "mystery story", "machine learning guide"]

print("Semantic Search Test Results")
print("=" * 60)

for query in test_queries:
    results = semantic_retriever.search(query, top_k=3)
    print(f"\nQuery: '{query}'")
    for rank, (doc_id, dist) in enumerate(results, 1):
        row = df.iloc[doc_id]
        book_title = lookup.get(row['parent_asin'], 'Unknown')
        print(f"  {rank}. Doc #{doc_id} - {book_title} - Distance: {dist:.4f}")

print("\nSemantic retriever working")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


FAISS index loaded from /Users/esteki/Desktop/MDS/575/DSCI_575_project_jchuang_esteki/data/processed/semantic_index
Semantic Search Test Results

Query: 'python book'
  1. Doc #8824 - Blippi: Let's See Animals! (8x8) - Distance: 0.8438
  2. Doc #6116 - Pizza Pig (Step into Reading) - Distance: 0.8928
  3. Doc #6100 - Hate Notes - Distance: 0.8996

Query: 'mystery story'
  1. Doc #1820 - The Da Vinci Code - Distance: 0.7629
  2. Doc #19153 - Crimes of Memory (A Detective Jackson Mystery) - Distance: 0.7751
  3. Doc #4735 - Gone Daddy Gone (Sloane Monroe Book 7) - Distance: 0.8295

Query: 'machine learning guide'
  1. Doc #11671 - GPS Mapping - Distance: 0.9625
  2. Doc #382 - Mastering Machine Learning with Python in Six Steps: A Practical Implementation Guide to Predictive Data Analytics Using Python - Distance: 1.0354
  3. Doc #9279 - Genetic Algorithms and Machine Learning for Programmers: Create AI Models and Evolve Solutions (Pragmatic Programmers) - Distance: 1.0635

Semantic retr